# Milestone 2: Exploratory Data Analysis (EDA) Notebook

This notebook analyzes the relationship between the cheap signal (draft next-token prediction entropy) and token acceptance, demonstrates the within-stream draft length variation of the controllers, and measures controller execution overhead.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

from src.serve.simulator import simulate_step, ACCEPT_PROBS
from src.controllers import EntropyThreshold, EpsilonGreedy, UCB, AcceptanceHistoryController

np.random.seed(42)
os.makedirs("results/figures", exist_ok=True)
print("Libraries and modules imported successfully.")

## 1. Analyzing Signal-Acceptance Correlation
We run the simulator for 5000 tokens on `spec_bench` and extract the prediction entropy and the target model's acceptance decision to verify correlation.

In [ ]:
rng = np.random.default_rng(42)
entropies = []
acceptances = []

# We simulate 1000 steps of length 8 to gather 8000 token decisions
for step in range(1000):
    # Using a dummy controller with fixed length 8
    res = simulate_step("spec_bench", 8, step, rng)
    entropies.extend(res["entropies"])
    acceptances.extend(res["acceptances"])

df_signal = pd.DataFrame({
    "entropy": entropies,
    "accepted": acceptances
})

# Bin the entropies and compute acceptance rate per bin
df_signal["entropy_bin"] = pd.cut(df_signal["entropy"], bins=np.arange(0.0, 3.1, 0.2))
bin_stats = df_signal.groupby("entropy_bin", observed=False)["accepted"].agg(["count", "mean"])

print("=== Signal-Acceptance Correlation (First 10 Bins) ===")
print(bin_stats.head(10))

# Plot and save the relationship
plt.figure(figsize=(8, 5))
valid_bins = bin_stats[bin_stats["count"] > 5]
bin_centers = [b.mid for b in valid_bins.index]
plt.plot(bin_centers, valid_bins["mean"], marker='o', color='purple', linewidth=2)
plt.title("Token Acceptance Rate vs. Next-Token Prediction Entropy")
plt.xlabel("Draft Next-Token Entropy (bits)")
plt.ylabel("Acceptance Rate")
plt.grid(True, linestyle='--', alpha=0.6)
plt.savefig("results/figures/fig2_entropy_correlation.png", dpi=300)
plt.close()

## 2. Within-Stream Draft Length Variation
We visualize how the EntropyThreshold controller and the Bandit controllers adapt the draft length step-by-step over a sequence of 100 steps.

In [ ]:
rng = np.random.default_rng(42)
entropy_ctrl = EntropyThreshold(tau=1.0, max_len=8)
lengths = []

for step in range(100):
    wl = rng.choice(list(ACCEPT_PROBS.keys()))
    res = simulate_step(wl, entropy_ctrl, step, rng)
    lengths.append(res["chosen_length"])

plt.figure(figsize=(12, 4))
plt.plot(lengths, marker='s', color='teal', label='EntropyThreshold (tau=1.0)')
plt.title("Step-by-Step Draft Length Decisions (Within-Sequence Adaptation)")
plt.xlabel("Simulation Step")
plt.ylabel("Chosen Draft Length (K)")
plt.ylim(0.5, 8.5)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.savefig("results/figures/fig3_length_variation.png", dpi=300)
plt.close()
print("Plot saved to results/figures/fig3_length_variation.png")

## 3. Controller Execution Overhead Measurement
We verify that the decision overhead of each controller is negligible (<0.1ms per step).

In [ ]:
import time

controllers = {
    "EntropyThreshold": EntropyThreshold(tau=1.0, max_len=8),
    "EpsilonGreedy": EpsilonGreedy(eps=0.1, seed=42),
    "UCB": UCB(c=0.5),
    "AcceptanceHistory": AcceptanceHistoryController()
}

print("=== Controller Selection Overhead per Step ===")
for name, ctrl in controllers.items():
    t_start = time.perf_counter()
    for step in range(10000):
        # Simulate choose/update
        if name == "EntropyThreshold":
            # Checks list of mock entropies
            ctrl.choose([0.2, 0.4, 0.7, 1.2, 0.5])
        else:
            k = ctrl.choose()
            if name == "AcceptanceHistory":
                ctrl.update(k, 3)
            else:
                ctrl.update(k, 1.5)
    elapsed = (time.perf_counter() - t_start) / 10000 * 1000  # ms per step
    print(f"Controller: {name:<20} | Average Overhead: {elapsed:.6f} ms (Target: < 0.1 ms)")